In [9]:
import pickle
import numpy as np

class CIFAR10Dataset:
    def __init__(self, dataset_dir):
        """
        Initialize the dataset class with the directory containing the dataset files.

        :param dataset_dir: Path to the directory containing the CIFAR-10 dataset files.
        """
        self.dataset_dir = dataset_dir
        self.label_names = None

    def unpickle(self, file):
        """
        Unpickle a given file and return its content as a dictionary.

        :param file: File path to unpickle.
        :return: A dictionary containing the unpickled data.
        """
        with open(file, 'rb') as fo:
            data_dict = pickle.load(fo, encoding='bytes')
        return data_dict

    def load_meta(self):
        """
        Load the metadata file to get label names.

        :return: A list of label names.
        """
        meta_file = f"{self.dataset_dir}/batches.meta"
        meta_dict = self.unpickle(meta_file)
        self.label_names = [label.decode('utf-8') for label in meta_dict[b'label_names']]

    def load_batch(self, batch_file):
        """
        Load a single batch file.

        :param batch_file: File path to the batch file.
        :return: A tuple of (data, labels).
        """
        batch_data = self.unpickle(batch_file)
        data = batch_data[b'data']
        labels = batch_data[b'labels']
        return data, labels

    def load_data(self):
        """
        Load all training data and the test set.

        :return: A tuple of (train_data, train_labels, test_data, test_labels).
        """
        train_data = []
        train_labels = []

        # Load training batches
        for i in range(1, 6):
            batch_file = f"{self.dataset_dir}/data_batch_{i}"
            data, labels = self.load_batch(batch_file)
            train_data.append(data)
            train_labels.extend(labels)

        train_data = np.vstack(train_data)
        train_labels = np.array(train_labels)

        # Reshape train data to (N, 32, 32, 3)
        train_data = train_data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        # Load test batch
        test_file = f"{self.dataset_dir}/test_batch"
        test_data, test_labels = self.load_batch(test_file)

        test_data = np.array(test_data)
        test_labels = np.array(test_labels)

        # Reshape test data to (N, 32, 32, 3)
        test_data = test_data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        return train_data, train_labels, test_data, test_labels

    def decode_image(self, image_data):
        """
        Decode a single row of image data into a 32x32x3 image.

        :param image_data: A 3072-element array representing the image data.
        :return: A 32x32x3 numpy array representing the image.
        """
        red = image_data[:1024].reshape(32, 32)
        green = image_data[1024:2048].reshape(32, 32)
        blue = image_data[2048:].reshape(32, 32)
        image = np.stack((red, green, blue), axis=-1)
        return image

    def label_to_name(self, label):
        """
        Convert a numeric label to its corresponding name.

        :param label: Numeric label.
        :return: The corresponding label name.
        """
        if self.label_names is None:
            raise ValueError("Label names have not been loaded. Call `load_meta` first.")
        return self.label_names[label]

    def save_as_npy(self, train_data, train_labels, test_data, test_labels, output_dir):
        """
        Save the CIFAR-10 dataset as .npy files with structured dictionaries.

        :param train_data: Training data array.
        :param train_labels: Training labels array.
        :param test_data: Test data array.
        :param test_labels: Test labels array.
        :param output_dir: Directory to save the .npy files.
        """
        labeled_data = {"images": train_data, "labels": train_labels}
        unlabeled_data = {"images": train_data, "labels": train_labels}  # Placeholder for unlabeled data
        test_data_dict = {"images": test_data, "labels": test_labels}

        np.save(f"{output_dir}/l_train.npy", labeled_data)
        np.save(f"{output_dir}/u_train.npy", unlabeled_data)
        np.save(f"{output_dir}/test.npy", test_data_dict)

if __name__ == "__main__":
    # Example usage
    dataset_dir = "./"
    output_dir = "../data"

    dataset = CIFAR10Dataset(dataset_dir)

    # Load metadata
    dataset.load_meta()
    print("Label names:", dataset.label_names)

    # Load data
    train_data, train_labels, test_data, test_labels = dataset.load_data()
    print("Training data shape:", train_data.shape)
    print("Training labels shape:", train_labels.shape)
    print("Test data shape:", test_data.shape)
    print("Test labels shape:", test_labels.shape)

    # Save as .npy files
    dataset.save_as_npy(train_data, train_labels, test_data, test_labels, output_dir)
    print(f".npy files saved in {output_dir}")


Label names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Training data shape: (50000, 32, 32, 3)
Training labels shape: (50000,)
Test data shape: (10000, 32, 32, 3)
Test labels shape: (10000,)
.npy files saved in ../data


In [10]:
l_train = np.load("../data/l_train.npy", allow_pickle=True).item()
print("Keys in l_train.npy:", l_train.keys())

Keys in l_train.npy: dict_keys(['images', 'labels'])


In [11]:
import numpy as np

# Load the labeled dataset
l_train = np.load("../data/l_train.npy", allow_pickle=True).item()
print("Shape of labeled images:", l_train["images"].shape)
print("Shape of labeled labels:", l_train["labels"].shape)

# Ensure all images are of shape (32, 32, 3)
for idx, img in enumerate(l_train["images"]):
    if img.shape != (32, 32, 3):
        print(f"Image {idx} has shape {img.shape}")


Shape of labeled images: (50000, 32, 32, 3)
Shape of labeled labels: (50000,)
